## Redrob Hackathon: Candidate Parsing and Feature Engineering

### What this notebook does
This notebook reads the raw candidate dataset and converts each profile into structured features for ranking.

### Why this notebook is needed
The ranking system cannot work directly on nested JSON. We first need to flatten the data and create meaningful features like experience, skill strength, behavioral readiness, and trust signals.

In [2]:
!pip install -q pandas numpy python-dateutil tqdm pyarrow

In [3]:
from pathlib import Path
import json
import re
from datetime import datetime
from dateutil import parser as date_parser
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
PROJECT_DIR = Path("/content/drive/MyDrive/Redrob_Hackathon")
DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"

CANDIDATES_PATH = DATA_DIR / "candidates.jsonl"
FLAGS_PATH = ARTIFACTS_DIR / "candidate_audit_with_flags.parquet"
JD_COMPASS_PATH = ARTIFACTS_DIR / "jd_compass.json"


In [6]:
print("Candidates exists:", CANDIDATES_PATH.exists())
print("Flags exists:", FLAGS_PATH.exists())
print("JD compass exists:", JD_COMPASS_PATH.exists())

Candidates exists: True
Flags exists: True
JD compass exists: True


In [7]:
# Load the candidate flags and JD compass
audit_df_flags = pd.read_parquet(FLAGS_PATH)

with open(JD_COMPASS_PATH, "r", encoding="utf-8") as f:
    jd_compass = json.load(f)

print("audit_df_flags shape:", audit_df_flags.shape)
print("JD compass keys:", jd_compass.keys())

audit_df_flags shape: (100000, 31)
JD compass keys: dict_keys(['jd_summary', 'skill_groups', 'scoring_dimensions'])


In [8]:
# Loading all the raw candidate data
def load_candidates(path):
    candidates = []
    with open(path, "r", encoding = "utf-8") as f:
      for line in tqdm(f, desc="loading candidates"):
        if line.strip():
          candidates.append(json.loads(line))

    return candidates

candidates = load_candidates(CANDIDATES_PATH)
print("Total candidates loaded:", len(candidates))
assert len(candidates) == audit_df_flags.shape[0], "Number of candidates and flags do not match"

loading candidates: 0it [00:00, ?it/s]

Total candidates loaded: 100000


In [9]:
from pprint import pprint
for key , value in candidates[0].items():
  pprint(f"{key}: {value}")

'candidate_id: CAND_0000001'
("profile: {'anonymized_name': 'Ira Vora', 'headline': 'Backend Engineer | "
 'SQL, Spark, Cloud\', \'summary\': "Software / data professional with 6.9 '
 'years of experience building data pipelines, backend systems, and analytics '
 "infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses "
 "are home territory; I'm building competence on the ML side. My toolkit is "
 'solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse '
 "design — and I've completed a couple of self-directed ML projects (Kaggle "
 'competitions, side projects fine-tuning small models). Interested in '
 'transitioning toward more AI/ML-focused work, ideally at a company where I '
 'can leverage my existing data-infra skills while learning modern ML '
 'practice.", \'location\': \'Toronto\', \'country\': \'Canada\', '
 "'years_of_experience': 6.9, 'current_title': 'Backend Engineer', "
 "'current_company': 'Mindtree', 'current_company_size': '100

In [10]:
'''
candidate data may contain missing values, extra spaces and different formating
so to bring all the data in same format we need to perform text preprocessing.
'''
# for safe text cleaning
def clean_text(value):
  if value is None:
    return ""
  value = str(value)
  value = value.strip()
  value = re.sub(r"\s+", " ", value)
  return value

In [11]:
# for safe numeric conversion
# Convert something into a float safely
def safe_float(value, default=0.0):
    try:
        if value is None:
            return default
        return float(value)
    except:
        return default

In [12]:
# Convert date strings into proper Python dates safely
def safe_parse_date(value):
    try:
        if value in [None, "", "null"]:
            return None
        return date_parser.parse(value).date()
    except:
        return None

In [13]:
rows = []

for candidate in tqdm(candidates, desc="Building feature rows"):
    profile = candidate.get("profile", {}) # If missing → return empty dictionary
    career_history = candidate.get("career_history", [])
    education = candidate.get("education", [])
    skills = candidate.get("skills", [])
    certifications = candidate.get("certifications", [])
    languages = candidate.get("languages", [])
    signals = candidate.get("redrob_signals", {})

    rows.append({
        "candidate_id": candidate.get("candidate_id"),
        "anonymized_name": clean_text(profile.get("anonymized_name")),
        "headline": clean_text(profile.get("headline")),
        "summary": clean_text(profile.get("summary")),
        "location": clean_text(profile.get("location")),
        "country": clean_text(profile.get("country")),
        "years_of_experience": safe_float(profile.get("years_of_experience")),
        "current_title": clean_text(profile.get("current_title")),
        "current_company": clean_text(profile.get("current_company")),
        "current_company_size": clean_text(profile.get("current_company_size")),
        "current_industry": clean_text(profile.get("current_industry")),

        "num_career_roles": len(career_history),
        "num_education": len(education),
        "num_skills": len(skills),
        "num_certifications": len(certifications),
        "num_languages": len(languages),

        "profile_completeness_score": safe_float(signals.get("profile_completeness_score")),
        "open_to_work_flag": int(bool(signals.get("open_to_work_flag"))),
        "recruiter_response_rate": safe_float(signals.get("recruiter_response_rate")),
        "avg_response_time_hours": safe_float(signals.get("avg_response_time_hours")),
        "notice_period_days": safe_float(signals.get("notice_period_days")),
        "github_activity_score": safe_float(signals.get("github_activity_score"), default=-1),
        "interview_completion_rate": safe_float(signals.get("interview_completion_rate")),
        "search_appearance_30d": safe_float(signals.get("search_appearance_30d")),
        "saved_by_recruiters_30d": safe_float(signals.get("saved_by_recruiters_30d")),
        "verified_email": int(bool(signals.get("verified_email"))),
        "verified_phone": int(bool(signals.get("verified_phone"))),
        "linkedin_connected": int(bool(signals.get("linkedin_connected"))),
    })

features_df = pd.DataFrame(rows)
print("Feature dataframe shape:", features_df.shape)
display(features_df.head())

Building feature rows:   0%|          | 0/100000 [00:00<?, ?it/s]

Feature dataframe shape: (100000, 28)


,candidate_id,anonymized_name,headline,summary,location,country,years_of_experience,current_title,current_company,current_company_size,...,recruiter_response_rate,avg_response_time_hours,notice_period_days,github_activity_score,interview_completion_rate,search_appearance_30d,saved_by_recruiters_30d,verified_email,verified_phone,linkedin_connected
0,CAND_0000001,Ira Vora,"Backend Engineer | SQL, Spark, Cloud",Software / data professional with 6.9 years of...,Toronto,Canada,6.9,Backend Engineer,Mindtree,10001+,...,0.34,177.8,60.0,9.2,0.71,249.0,4.0,1,1,0
1,CAND_0000002,Saanvi Sethi,Operations Manager | 12.5+ yrs experience,Professional with 12.5+ years of experience. M...,"Chennai, Tamil Nadu",India,12.5,Operations Manager,Wipro,10001+,...,0.29,171.6,60.0,-1.0,0.62,107.0,10.0,0,0,0
2,CAND_0000003,Yash Agarwal,Customer Support | 1.1+ yrs experience,Professional with 1.1+ years of experience. I'...,Austin,USA,1.1,Customer Support,TCS,10001+,...,0.46,119.4,150.0,-1.0,0.86,28.0,4.0,1,0,0
3,CAND_0000004,Anil Bose,Marketing Manager | Driving business outcomes,Professional with 3.8+ years of experience. My...,Sydney,Australia,3.8,Marketing Manager,Dunder Mifflin,201-500,...,0.26,104.1,120.0,-1.0,0.35,5.0,8.0,1,1,1
4,CAND_0000005,Aisha Sethi,Accountant | Helping teams scale,Professional with 11.0+ years of experience. I...,"Gurgaon, Haryana",India,11.0,Accountant,Stark Industries,1001-5000,...,0.37,116.7,30.0,-1.0,0.74,67.0,1.0,0,1,1


In [14]:
#  career history analysis
def extract_skill_names(skill_list):
    if not isinstance(skill_list, list):
        return []
    return [clean_text(skill.get("name")) for skill in skill_list if clean_text(skill.get("name"))]

In [15]:
skill_texts = []

for candidate in candidates:
    skills = candidate.get("skills", [])
    skill_names = extract_skill_names(skills)
    skill_texts.append(" | ".join(skill_names))

features_df["skill_text"] = skill_texts
display(features_df[["candidate_id", "skill_text"]].head())

,candidate_id,skill_text
0,CAND_0000001,Tailwind | NLP | Image Classification | Fine-t...
1,CAND_0000002,Project Management | React | Photoshop | TypeS...
2,CAND_0000003,Angular | SEO | Excel | Accounting | Kubernete...
3,CAND_0000004,Node.js | Content Writing | Redux | Airflow | ...
4,CAND_0000005,SQL | PowerPoint | Photoshop | Tailwind | Apac...


In [16]:
profile_texts = []

for candidate in candidates:
    profile = candidate.get("profile", {})
    career_history = candidate.get("career_history", [])

    career_parts = []
    for role in career_history:
        title = clean_text(role.get("title"))
        company = clean_text(role.get("company"))
        industry = clean_text(role.get("industry"))
        desc = clean_text(role.get("description"))
        career_parts.append(f"{title} at {company} in {industry}. {desc}")

    profile_text = " ".join([
        clean_text(profile.get("headline")),
        clean_text(profile.get("summary")),
        clean_text(profile.get("current_title")),
        clean_text(profile.get("current_company")),
        clean_text(profile.get("current_industry")),
        " ".join(career_parts)
    ])

    profile_texts.append(profile_text)

features_df["profile_text"] = profile_texts
display(features_df[["candidate_id", "profile_text"]].head())

,candidate_id,profile_text
0,CAND_0000001,"Backend Engineer | SQL, Spark, Cloud Software ..."
1,CAND_0000002,Operations Manager | 12.5+ yrs experience Prof...
2,CAND_0000003,Customer Support | 1.1+ yrs experience Profess...
3,CAND_0000004,Marketing Manager | Driving business outcomes ...
4,CAND_0000005,Accountant | Helping teams scale Professional ...


In [17]:
technical_keywords = [
    "python", "ml", "machine learning", "llm", "rag", "retrieval", "ranking",
    "vector", "faiss", "elasticsearch", "pytorch", "tensorflow", "langchain",
    "langgraph", "embeddings", "search", "nlp", "recommendation"
]

def count_technical_keywords(text):
    text = clean_text(text).lower()
    count = 0
    for keyword in technical_keywords:
        if keyword in text:
            count += 1
    return count

features_df["technical_keyword_count"] = features_df["profile_text"].apply(count_technical_keywords)
display(features_df[["candidate_id", "technical_keyword_count"]].head())

,candidate_id,technical_keyword_count
0,CAND_0000001,3
1,CAND_0000002,3
2,CAND_0000003,0
3,CAND_0000004,3
4,CAND_0000005,0


In [18]:
flags_subset = audit_df_flags[
    [
        "candidate_id",
        "hard_risk_count",
        "soft_risk_count",
        "total_risk_score",
        "flag_many_skills",
        "flag_low_experience_many_roles",
        "flag_bad_profile_completeness",
        "flag_very_low_response_rate",
        "flag_very_slow_response",
        "flag_long_notice_period",
        "flag_no_github_for_technical_profile",
        "flag_low_interview_completion",
    ]
]

features_df = features_df.merge(flags_subset, on="candidate_id", how="left")
print("After merge shape:", features_df.shape)
display(features_df.head())

After merge shape: (100000, 42)


,candidate_id,anonymized_name,headline,summary,location,country,years_of_experience,current_title,current_company,current_company_size,...,soft_risk_count,total_risk_score,flag_many_skills,flag_low_experience_many_roles,flag_bad_profile_completeness,flag_very_low_response_rate,flag_very_slow_response,flag_long_notice_period,flag_no_github_for_technical_profile,flag_low_interview_completion
0,CAND_0000001,Ira Vora,"Backend Engineer | SQL, Spark, Cloud",Software / data professional with 6.9 years of...,Toronto,Canada,6.9,Backend Engineer,Mindtree,10001+,...,0,0,False,False,False,False,False,False,False,False
1,CAND_0000002,Saanvi Sethi,Operations Manager | 12.5+ yrs experience,Professional with 12.5+ years of experience. M...,"Chennai, Tamil Nadu",India,12.5,Operations Manager,Wipro,10001+,...,0,0,False,False,False,False,False,False,False,False
2,CAND_0000003,Yash Agarwal,Customer Support | 1.1+ yrs experience,Professional with 1.1+ years of experience. I'...,Austin,USA,1.1,Customer Support,TCS,10001+,...,1,3,False,False,True,False,False,True,False,False
3,CAND_0000004,Anil Bose,Marketing Manager | Driving business outcomes,Professional with 3.8+ years of experience. My...,Sydney,Australia,3.8,Marketing Manager,Dunder Mifflin,201-500,...,2,4,False,False,True,False,False,True,False,True
4,CAND_0000005,Aisha Sethi,Accountant | Helping teams scale,Professional with 11.0+ years of experience. I...,"Gurgaon, Haryana",India,11.0,Accountant,Stark Industries,1001-5000,...,0,0,False,False,False,False,False,False,False,False


In [19]:
numeric_cols = features_df.select_dtypes(include=["int64", "float64"]).columns

for col in numeric_cols:
    features_df[col] = features_df[col].fillna(0)

print("Missing values after fill:")
display(features_df.isna().sum().sort_values(ascending=False).head(20))

Missing values after fill:


,0
candidate_id,0
anonymized_name,0
headline,0
summary,0
location,0
country,0
years_of_experience,0
current_title,0
current_company,0
current_company_size,0


In [20]:
features_df["signal_strength"] = (
    features_df["profile_completeness_score"] * 0.3 +
    features_df["recruiter_response_rate"] * 100 * 0.2 +
    features_df["interview_completion_rate"] * 100 * 0.2 +
    features_df["saved_by_recruiters_30d"] * 0.1 +
    features_df["technical_keyword_count"] * 5
)

features_df["availability_strength"] = (
    features_df["open_to_work_flag"] * 20 +
    (1 / (1 + features_df["notice_period_days"])) * 50 +
    features_df["recruiter_response_rate"] * 30
)

display(features_df[[
    "candidate_id",
    "signal_strength",
    "availability_strength"
]].head())

,candidate_id,signal_strength,availability_strength
0,CAND_0000001,62.47,31.019672
1,CAND_0000002,57.81,29.519672
2,CAND_0000003,36.37,14.131126
3,CAND_0000004,36.55,8.213223
4,CAND_0000005,47.68,32.712903


In [21]:
summary_cols = [
    "candidate_id",
    "years_of_experience",
    "num_career_roles",
    "num_skills",
    "profile_completeness_score",
    "recruiter_response_rate",
    "avg_response_time_hours",
    "notice_period_days",
    "github_activity_score",
    "interview_completion_rate",
    "technical_keyword_count",
    "hard_risk_count",
    "soft_risk_count",
    "total_risk_score",
    "signal_strength",
    "availability_strength",
]

display(features_df[summary_cols].head(20))

,candidate_id,years_of_experience,num_career_roles,num_skills,profile_completeness_score,recruiter_response_rate,avg_response_time_hours,notice_period_days,github_activity_score,interview_completion_rate,technical_keyword_count,hard_risk_count,soft_risk_count,total_risk_score,signal_strength,availability_strength
0,CAND_0000001,6.9,2,17,86.9,0.34,177.8,60.0,9.2,0.71,3,0,0,0,62.47,31.019672
1,CAND_0000002,12.5,4,9,78.7,0.29,171.6,60.0,-1.0,0.62,3,0,0,0,57.81,29.519672
2,CAND_0000003,1.1,1,6,31.9,0.46,119.4,150.0,-1.0,0.86,0,1,1,3,36.37,14.131126
3,CAND_0000004,3.8,3,10,28.5,0.26,104.1,120.0,-1.0,0.35,3,1,2,4,36.55,8.213223
4,CAND_0000005,11.0,4,6,84.6,0.37,116.7,30.0,-1.0,0.74,0,0,0,0,47.68,32.712903
5,CAND_0000006,6.0,2,8,29.7,0.12,172.1,150.0,-1.0,0.57,0,2,1,5,23.61,3.931126
6,CAND_0000007,5.5,2,7,74.6,0.62,61.3,30.0,-1.0,0.47,0,0,2,2,44.98,20.212903
7,CAND_0000008,3.6,1,8,63.0,0.42,98.4,90.0,-1.0,0.74,0,0,0,0,42.10,13.149451
8,CAND_0000009,11.0,4,6,39.6,0.53,202.0,150.0,-1.0,0.54,0,0,2,2,33.38,16.231126
9,CAND_0000010,4.6,1,17,81.6,0.40,19.0,120.0,33.7,0.53,3,0,1,1,58.28,12.413223


In [22]:
FEATURES_PATH = ARTIFACTS_DIR / "candidate_features.parquet"
FEATURES_CSV_PATH = ARTIFACTS_DIR / "candidate_features.csv"

features_df.to_parquet(FEATURES_PATH, index=False)
features_df.to_csv(FEATURES_CSV_PATH, index=False)

print("Saved:", FEATURES_PATH)
print("Saved:", FEATURES_CSV_PATH)

Saved: /content/drive/MyDrive/Redrob_Hackathon/artifacts/candidate_features.parquet
Saved: /content/drive/MyDrive/Redrob_Hackathon/artifacts/candidate_features.csv


In [23]:
summary = {
    "rows": int(features_df.shape[0]),
    "columns": int(features_df.shape[1]),
    "avg_years_of_experience": float(features_df["years_of_experience"].mean()),
    "avg_profile_completeness": float(features_df["profile_completeness_score"].mean()),
    "avg_total_risk_score": float(features_df["total_risk_score"].mean()),
    "num_candidates_open_to_work": int(features_df["open_to_work_flag"].sum()),
}

with open(ARTIFACTS_DIR / "candidate_features_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

print(summary)

{'rows': 100000, 'columns': 44, 'avg_years_of_experience': 7.166318999999999, 'avg_profile_completeness': 56.75818, 'avg_total_risk_score': 1.56649, 'num_candidates_open_to_work': 35339}


The previous cells created general candidate features and now we will create role-specific features based on the Senior AI Engineer JD.
- retrieval and ranking evidence
- production engineering evidence
- evaluation evidence
- seniority fit
- location fit
- possible disqualifier signals

In [24]:
skill_groups = jd_compass["skill_groups"]
jd_summary = jd_compass["jd_summary"]

print("JD role:", jd_summary["role_title"])
print("Skill groups:", list(skill_groups.keys()))

JD role: Senior AI Engineer
Skill groups: ['core_ai_ml', 'retrieval_ranking', 'production_engineering', 'evaluation_and_quality']


In [25]:
def count_group_matches(text, keywords):
    text = clean_text(text).lower()

    matched_keywords = []

    for keyword in keywords:
        if keyword.lower() in text:
            matched_keywords.append(keyword)

    return len(set(matched_keywords))

In [26]:
features_df["core_ai_ml_match_count"] = features_df["profile_text"].apply(
    lambda text: count_group_matches(text, skill_groups["core_ai_ml"])
)

features_df["retrieval_ranking_match_count"] = features_df["profile_text"].apply(
    lambda text: count_group_matches(text, skill_groups["retrieval_ranking"])
)

features_df["production_engineering_match_count"] = features_df["profile_text"].apply(
    lambda text: count_group_matches(text, skill_groups["production_engineering"])
)

features_df["evaluation_quality_match_count"] = features_df["profile_text"].apply(
    lambda text: count_group_matches(text, skill_groups["evaluation_and_quality"])
)

display(
    features_df[
        [
            "candidate_id",
            "core_ai_ml_match_count",
            "retrieval_ranking_match_count",
            "production_engineering_match_count",
            "evaluation_quality_match_count",
        ]
    ].head(10)
)

,candidate_id,core_ai_ml_match_count,retrieval_ranking_match_count,production_engineering_match_count,evaluation_quality_match_count
0,CAND_0000001,0,1,1,0
1,CAND_0000002,1,0,1,0
2,CAND_0000003,0,0,0,0
3,CAND_0000004,1,0,1,0
4,CAND_0000005,0,0,0,0
5,CAND_0000006,0,0,0,0
6,CAND_0000007,0,0,1,0
7,CAND_0000008,0,0,0,0
8,CAND_0000009,0,0,1,1
9,CAND_0000010,0,1,1,1


In [27]:
features_df["seniority_fit_score"] = 0.0

features_df.loc[
    features_df["years_of_experience"].between(5, 9, inclusive="both"),
    "seniority_fit_score"
] = 1.0

features_df.loc[
    features_df["years_of_experience"].between(4, 10, inclusive="both") &
    (features_df["seniority_fit_score"] == 0),
    "seniority_fit_score"
] = 0.6

features_df.loc[
    features_df["years_of_experience"].between(3, 12, inclusive="both") &
    (features_df["seniority_fit_score"] == 0),
    "seniority_fit_score"
] = 0.3

display(
    features_df[
        ["candidate_id", "years_of_experience", "seniority_fit_score"]
    ].head(10)
)

,candidate_id,years_of_experience,seniority_fit_score
0,CAND_0000001,6.9,1.0
1,CAND_0000002,12.5,0.0
2,CAND_0000003,1.1,0.0
3,CAND_0000004,3.8,0.3
4,CAND_0000005,11.0,0.3
5,CAND_0000006,6.0,1.0
6,CAND_0000007,5.5,1.0
7,CAND_0000008,3.6,0.3
8,CAND_0000009,11.0,0.3
9,CAND_0000010,4.6,0.6


In [28]:
preferred_locations = [
    "pune",
    "noida",
    "delhi",
    "gurgaon",
    "gurugram",
    "mumbai",
    "hyderabad",
    "bangalore",
    "bengaluru",
]

features_df["location_fit_flag"] = features_df["location"].apply(
    lambda location: int(
        any(city in clean_text(location).lower() for city in preferred_locations)
    )
)

display(
    features_df[
        ["candidate_id", "location", "location_fit_flag"]
    ].head(10)
)

,candidate_id,location,location_fit_flag
0,CAND_0000001,Toronto,0
1,CAND_0000002,"Chennai, Tamil Nadu",0
2,CAND_0000003,Austin,0
3,CAND_0000004,Sydney,0
4,CAND_0000005,"Gurgaon, Haryana",1
5,CAND_0000006,Austin,0
6,CAND_0000007,"Gurgaon, Haryana",1
7,CAND_0000008,"Noida, Uttar Pradesh",1
8,CAND_0000009,New York,0
9,CAND_0000010,London,0


In [29]:
features_df["flag_possible_framework_only_profile"] = (
    features_df["profile_text"].str.lower().str.contains(
        "langchain|openai",
        regex=True,
        na=False
    )
    &
    (features_df["retrieval_ranking_match_count"] <= 1)
    &
    (features_df["production_engineering_match_count"] <= 1)
)

features_df["flag_possible_non_ir_ai_profile"] = (
    features_df["profile_text"].str.lower().str.contains(
        "computer vision|cv engineer|speech|robotics",
        regex=True,
        na=False
    )
    &
    (features_df["retrieval_ranking_match_count"] == 0)
)

display(
    features_df[
        [
            "candidate_id",
            "flag_possible_framework_only_profile",
            "flag_possible_non_ir_ai_profile",
        ]
    ].head(20)
)

,candidate_id,flag_possible_framework_only_profile,flag_possible_non_ir_ai_profile
0,CAND_0000001,False,False
1,CAND_0000002,False,False
2,CAND_0000003,False,False
3,CAND_0000004,False,False
4,CAND_0000005,False,False
5,CAND_0000006,False,False
6,CAND_0000007,False,False
7,CAND_0000008,False,False
8,CAND_0000009,False,False
9,CAND_0000010,False,False


In [30]:
features_df["role_evidence_score"] = (
    features_df["core_ai_ml_match_count"] * 1.0
    + features_df["retrieval_ranking_match_count"] * 2.0
    + features_df["production_engineering_match_count"] * 1.5
    + features_df["evaluation_quality_match_count"] * 2.0
    + features_df["seniority_fit_score"] * 3.0
    + features_df["location_fit_flag"] * 0.5
    - features_df["flag_possible_framework_only_profile"].astype(int) * 2.0
    - features_df["flag_possible_non_ir_ai_profile"].astype(int) * 2.0
)

display(
    features_df[
        [
            "candidate_id",
            "role_evidence_score",
            "retrieval_ranking_match_count",
            "production_engineering_match_count",
            "evaluation_quality_match_count",
            "seniority_fit_score",
            "total_risk_score",
        ]
    ]
    .sort_values("role_evidence_score", ascending=False)
    .head(20)
)

,candidate_id,role_evidence_score,retrieval_ranking_match_count,production_engineering_match_count,evaluation_quality_match_count,seniority_fit_score,total_risk_score
5648,CAND_0005649,32.5,8,2,4,1.0,0
79386,CAND_0079387,32.0,8,2,4,1.0,2
44854,CAND_0044855,32.0,8,2,4,1.0,0
83306,CAND_0083307,32.0,8,2,4,1.0,3
69904,CAND_0069905,32.0,8,2,4,1.0,0
30952,CAND_0030953,32.0,8,2,4,1.0,0
55904,CAND_0055905,31.5,9,1,3,1.0,1
81845,CAND_0081846,31.5,9,1,3,1.0,2
41610,CAND_0041611,31.5,8,1,4,1.0,5
58574,CAND_0058575,30.0,7,2,4,1.0,0


In [31]:
top_role_evidence = features_df.sort_values(
    "role_evidence_score",
    ascending=False
).head(10)

display(
    top_role_evidence[
        [
            "candidate_id",
            "current_title",
            "current_company",
            "location",
            "years_of_experience",
            "role_evidence_score",
            "retrieval_ranking_match_count",
            "production_engineering_match_count",
            "evaluation_quality_match_count",
            "total_risk_score",
        ]
    ]
)

,candidate_id,current_title,current_company,location,years_of_experience,role_evidence_score,retrieval_ranking_match_count,production_engineering_match_count,evaluation_quality_match_count,total_risk_score
5648,CAND_0005649,Senior Data Scientist,Sarvam AI,"Delhi, Delhi",7.4,32.5,8,2,4,0
79386,CAND_0079387,AI Engineer,Microsoft,"Trivandrum, Kerala",6.9,32.0,8,2,4,2
44854,CAND_0044855,Senior Data Scientist,Flipkart,"Coimbatore, Tamil Nadu",6.6,32.0,8,2,4,0
83306,CAND_0083307,Search Engineer,CRED,"Vizag, Andhra Pradesh",7.8,32.0,8,2,4,3
69904,CAND_0069905,Applied ML Engineer,Sarvam AI,"Bhubaneswar, Odisha",6.6,32.0,8,2,4,0
30952,CAND_0030953,Search Engineer,Nykaa,"Chennai, Tamil Nadu",7.8,32.0,8,2,4,0
55904,CAND_0055905,Senior Machine Learning Engineer,Flipkart,London,8.1,31.5,9,1,3,1
81845,CAND_0081846,Lead AI Engineer,Razorpay,"Jaipur, Rajasthan",6.7,31.5,9,1,3,2
41610,CAND_0041611,Staff Machine Learning Engineer,Locobuzz,Austin,6.4,31.5,8,1,4,5
58574,CAND_0058575,AI Engineer,Krutrim,"Chennai, Tamil Nadu",5.8,30.0,7,2,4,0


In [32]:
features_df.to_parquet(
    ARTIFACTS_DIR / "candidate_features.parquet",
    index=False
)

features_df.to_csv(
    ARTIFACTS_DIR / "candidate_features.csv",
    index=False
)

print("Updated candidate feature files saved.")
print("Final feature shape:", features_df.shape)

Updated candidate feature files saved.
Final feature shape: (100000, 53)
